# Solar Filament Segmentation 2026 — H9b reproducibility audit

**Author:** Francisco Angulo de Lafuente · ORCID: 0009-0001-1634-7063

**Kaggle submission:** 58112557 · public PQ **0.39**, 8 November 2026.

This notebook verifies the original candidate's **H6 selection and H9b
native-mask assembly** deterministically, starting from frozen outputs of the
four U-Nets, Mask R-CNN proposal detector and two crop-refinement models.
It loads only model-produced predictions from the official test H-alpha images;
no ground-truth annotations, observatory labels, temporal neighbours or public
test-answer files are used.

**Reproducibility boundary:** The stage-1 probability maps, detector proposals
and native crop probabilities are included as fixed intermediate evidence.
The notebook does **not** recompute those predictions from the raw test
images; a fully independent end-to-end model inference requires running the
upstream model implementations and weights as well. This is a transparent
release-audit notebook, not proof of full raw-image reproducibility.


In [ ]:
from __future__ import annotations
import csv, hashlib, json, os, pathlib, subprocess, sys, time
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src" / "h9_official.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "src" / "h9_official.py").is_file(), "Open this notebook from the release root or notebooks directory"
ARTIFACT = ROOT / "submission_58112557.csv"
EXPECTED_SHA = "67f02e4d54b78424306133840ec144b426c8b308f6fc659f90a7f67131d5a79e"
def sha(path: pathlib.Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for data in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(data)
    return h.hexdigest()
assert sha(ARTIFACT) == EXPECTED_SHA, "Frozen Kaggle CSV has changed"
print("FROZEN_SUBMISSION_OK", ROOT, EXPECTED_SHA)


## 1. Verify the frozen, label-free prediction intermediates

The four ResNet34 U-Net probability maps, Mask R-CNN proposals, H6 component
selection table, two final crop-refiner predictions and their affine frames
are stored with the release. The frozen network weights and inference source
are retained separately. All fixed output policies were chosen without
consulting the public Kaggle leaderboard.


In [ ]:
required = [
    ROOT / "dets_official.pkl",
    ROOT / "h6/components.pkl",
    ROOT / "submission.csv",
    ROOT / "h9/official/probs_resnet34.npz",
    ROOT / "h9/official/probs_efficientnet_v2_s.npz",
    ROOT / "models/refiner_ALL.pt",
    ROOT / "models/refiner_ALL_efficientnet_v2_s.pt",
    ROOT / "src/h9_official.py",
    ROOT / "src/h6_official.py",
    ROOT / "src/h9_score.py",
    ROOT / "filseg/postprocess.py",
]
required.extend(
    ROOT / f"models/{block}_resnet34_s0_fp16.pt"
    for block in ("F1", "F2", "F3", "F4")
)
assert all(path.is_file() and path.stat().st_size > 0 for path in required), [
    str(path) for path in required if not path.is_file()
]
crop_files = sorted((ROOT / "h9/official/crops").glob("*.npz"))
assert len(crop_files) == 180, len(crop_files)
for block in ("F1", "F2", "F3", "F4"):
    assert len(list((ROOT / f"h3/maps/{block}_resnet34_s0/official").glob("*.png"))) == 180
print("INPUT_MANIFEST_VERIFIED", len(required), "fixed files,", len(crop_files), "image crops")


## 2. Replay H6 selection and both H9b refiner outputs

This command executes the original inference-stage assembly code with the
fixed H6 component selector and both independently trained refiner outputs.
The intermediate tensors are predictions, not labels. Each final mask is
projected back to the original **2048 × 2048** coordinate system with a
recorded affine transform; the output is a standard, disjoint COCO-RLE CSV.


In [ ]:
output = ROOT / "rebuild_notebook"
(output / "crops").mkdir(parents=True, exist_ok=True)
import shutil
for source in crop_files:
    destination = output / "crops" / source.name
    if not destination.exists():
        shutil.copyfile(source, destination)
start = time.monotonic()
environment = os.environ.copy()
environment["SOLARFIL_ASSET_ROOT"] = str(ROOT)
environment["PYTHONPATH"] = os.pathsep.join(
    [str(ROOT), str(ROOT / "src"), environment.get("PYTHONPATH", "")]
)
command = [
    sys.executable, str(ROOT / "src/h9_official.py"), "paste",
    "--out", str(output),
    "--probs",
    str(ROOT / "h9/official/probs_resnet34.npz"),
    str(ROOT / "h9/official/probs_efficientnet_v2_s.npz"),
]
subprocess.run(command, cwd=ROOT, env=environment, check=True)
generated = output / "submission_h9b.csv"
assert generated.is_file(), "No output CSV produced"
elapsed = time.monotonic() - start
print("ASSEMBLY_SECONDS", round(elapsed, 2))


## 3. Validate the exact original 0.39 submission

The strongest check is **byte-for-byte identity**, not a visually similar
CSV. The recorded Kaggle entry contains 1,241 filament rows covering all
180 official test images and no duplicated filament identifiers.


In [ ]:
actual_sha = sha(generated)
assert actual_sha == EXPECTED_SHA, (
    f"Replay mismatch: {actual_sha} vs official SHA {EXPECTED_SHA}"
)
with generated.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    assert reader.fieldnames == ["filament_id", "segmentation_rle"]
    rows = list(reader)
ids = [row["filament_id"] for row in rows]
stems = {name.rsplit("_", 1)[0] for name in ids}
assert len(rows) == len(ids) == len(set(ids)) == 1241
assert len(stems) == 180
assert all(row["segmentation_rle"] for row in rows)
print("EXACT_REPLAY_PASS", actual_sha, len(rows), "filaments", len(stems), "images")


## 4. Publish the audit receipt

The JSON receipt records the precise reproducibility boundary and hashes.
It is intentionally distinct from a Kaggle submission confirmation.
The original submission score **0.39** is independently verified by the
Kaggle CLI (ID 58112557), but this notebook does not contact Kaggle.


In [ ]:
audit = {
    "status": "H9B_STAGE2_EXACT_REPLAY_PASS",
    "scope": "Frozen model-output cache -> H6 selection -> H9b assembly -> CSV",
    "raw_image_end_to_end_verified": False,
    "official_kaggle_submission_id": 58112557,
    "official_public_pq_historical": 0.39,
    "replayed_file": generated.name,
    "sha256": actual_sha,
    "rows": len(rows),
    "images": len(stems),
    "staged_checkpoint_count": 6,
    "replay_seconds": round(elapsed, 2),
    "test_ground_truth_used": False,
    "external_labels_used": False,
}
receipt = output / "audit_receipt.json"
receipt.write_text(json.dumps(audit, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(audit, indent=2, sort_keys=True))
